# Text to 3D AI Model - Colab + ngrok

This notebook runs the GitHub project on a Colab GPU and exposes the FastAPI frontend/backend through ngrok.

## 1. Clone the GitHub repository

In [ ]:
REPO_URL = "https://github.com/LucyAlex12/Text_to_3d_ai_model.git"

!rm -rf Text_to_3d_ai_model
!git clone {REPO_URL}
%cd Text_to_3d_ai_model

In [ ]:
import sys
import os

TRIPOSR_PATH = "/content/Text_to_3d_ai_model/TripoSR"

if TRIPOSR_PATH not in sys.path:
    sys.path.append(TRIPOSR_PATH)

print("Added TripoSR to sys.path")
print(sys.path[-1])

## 2. Install dependencies

If Colab asks you to restart the runtime after installs, restart it, then rerun the cells from the top.

In [ ]:
# Clean conflicting packages first
!pip uninstall -y cupy cupy-cuda12x cupy-cuda11x numpy pymatting rembg

# Stable numpy for Colab + torch ecosystem
!pip install numpy==1.26.4

# Install backend dependencies
!pip install rembg==2.0.59 pymatting==1.1.12
!pip install fastapi uvicorn python-multipart pyngrok pillow huggingface_hub
!pip install transformers diffusers accelerate safetensors
!pip install trimesh xatlas pygltflib pyyaml omegaconf==2.3.0 einops==0.7.0
!pip install git+https://github.com/tatsy/torchmcubes.git

## 3. Download TripoSR weights

`TripoSR/model.ckpt` is large, so the notebook downloads it from Hugging Face instead of storing it in Git.

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

# Replace the incomplete TripoSR folder from Git with the full source code.
!rm -rf TripoSR
!git clone https://github.com/VAST-AI-Research/TripoSR.git TripoSR

if not Path("TripoSR/tsr/system.py").exists():
    raise RuntimeError("TripoSR source clone failed: missing TripoSR/tsr/system.py")

# Download required model files into the same TripoSR folder.
for filename in ["config.yaml", "model.ckpt"]:
    hf_hub_download(
        repo_id="stabilityai/TripoSR",
        filename=filename,
        local_dir="TripoSR",
        local_dir_use_symlinks=False
    )

print("TripoSR source + checkpoint ready")

## 4. Configure ngrok

Create an ngrok authtoken at https://dashboard.ngrok.com/get-started/your-authtoken.

The next cell uses `getpass()` instead of Colab Secrets because Colab's secret vault can fail with `Failed to fetch` / `await connected: disconnected` errors.

Optional: if you reserved an ngrok static domain, enter it when prompted, for example `your-name.ngrok-free.app`.

In [ ]:
from getpass import getpass
from pyngrok import ngrok

NGROK_AUTH_TOKEN = getpass("Paste your ngrok authtoken: ").strip()
NGROK_STATIC_DOMAIN = input("Optional ngrok static domain, or press Enter: ").strip()

if not NGROK_AUTH_TOKEN:
    raise ValueError("ngrok authtoken is required")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

if NGROK_STATIC_DOMAIN:
    print("ngrok configured with static domain:", NGROK_STATIC_DOMAIN)
else:
    print("ngrok configured with a temporary URL")

## 5. Start backend and expose the app

Open the printed ngrok URL in a normal browser tab. Do not use Colab's iframe preview.

In [ ]:
%cd /content/Text_to_3d_ai_model

import os
import subprocess
import sys
import time
from pathlib import Path

import requests
from pyngrok import ngrok

TRIPOSR_PATH = "/content/Text_to_3d_ai_model/TripoSR"
if not Path(TRIPOSR_PATH, "tsr", "system.py").exists():
    raise RuntimeError("Missing TripoSR source. Rerun section 3 before starting the backend.")

if TRIPOSR_PATH not in sys.path:
    sys.path.insert(0, TRIPOSR_PATH)
import tsr
print("Verified tsr import from:", tsr.__file__)

os.environ["IMAGE_MODEL_KIND"] = "sd15"
os.environ["SDXL_WIDTH"] = "512"
os.environ["SDXL_HEIGHT"] = "512"
os.environ["SDXL_STEPS"] = "20"
os.environ["SDXL_GUIDANCE_SCALE"] = "7.0"
os.environ["TRIPOSR_MAX_MC_RESOLUTION"] = "256"

server_env = os.environ.copy()
server_env["PYTHONPATH"] = TRIPOSR_PATH + os.pathsep + server_env.get("PYTHONPATH", "")

ngrok.kill()

server = subprocess.Popen(
    [
        "python",
        "-c",
        "import sys; sys.path.insert(0, '/content/Text_to_3d_ai_model/TripoSR'); "
        "import uvicorn; uvicorn.run('api:app', host='0.0.0.0', port=8000)"
    ],
    cwd="/content/Text_to_3d_ai_model",
    env=server_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print("Starting backend. This can take a few minutes while models load...")

for _ in range(240):
    if server.poll() is not None:
        remaining = server.stdout.read() if server.stdout else ""
        raise RuntimeError("Backend stopped before it was ready. Logs:\n" + remaining[-4000:])
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.ok:
            print("Backend is ready")
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Backend did not become ready. Run the logs cell below.")

if NGROK_STATIC_DOMAIN:
    tunnel = ngrok.connect(8000, "http", domain=NGROK_STATIC_DOMAIN)
else:
    tunnel = ngrok.connect(8000, "http")

public_url = tunnel.public_url

print("\nOpen this public app URL:")
print(public_url)
print("\nKeep this Colab runtime running while using the app.")
print("If ngrok shows a browser warning page, click through once. The app also sends the ngrok-skip-browser-warning header for API/model requests.")

## Optional: view backend logs

Run this only if the app fails or you want to monitor generation logs.

In [ ]:
while True:
    line = server.stdout.readline()
    if not line:
        break
    print(line, end="")